<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo"  />
    </a>
</p>


# **Finding Correlation**


Estimated time needed: **30** minutes


In this lab, you will work with a cleaned dataset to perform exploratory data analysis (EDA). You will examine the distribution of the data, identify outliers, and determine the correlation between different columns in the dataset.


## Objectives


In this lab, you will perform the following:


- Identify the distribution of compensation data in the dataset.

- Remove outliers to refine the dataset.

- Identify correlations between various features in the dataset.


## Hands on Lab


##### Step 1: Install and Import Required Libraries


In [ ]:


# Import libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

### Step 2: Load the Dataset


In [ ]:
import os

# Use local file path instead of remote URL
file_path = "survey-data.csv"

# Check if local file exists
if not os.path.exists(file_path):
    print(f"Warning: Local file '{file_path}' not found!")
    print("Please download the survey data file to your working directory.")
else:
    file_size = os.path.getsize(file_path) / 1024 / 1024
    print(f"Using local file: '{file_path}' ({file_size:.2f} MB)")

# Create the dataframe from local file
df = pd.read_csv(file_path)

# OLD CODE (commented out - was using remote URL):
# file_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/n01PQ9pSmiRX6520flujwQ/survey-data.csv"
# df = pd.read_csv(file_url)

#Display the top 10 records
df.head()

<h3>Step 3: Analyze and Visualize Compensation Distribution</h3>


**Task**: Plot the distribution and histogram for `ConvertedCompYearly` to examine the spread of yearly compensation among respondents.


In [ ]:
# Step 3: Analyze and Visualize Compensation Distribution (Improved Version)

# Fill missing values with random sampling to avoid clustering
valid_values = df['ConvertedCompYearly'].dropna()
mask = df['ConvertedCompYearly'].isna()
df.loc[mask, 'ConvertedCompYearly'] = np.random.choice(valid_values, size=mask.sum())

# Create two subplots: Log-Transformed Histogram and Distribution (Density) Plot
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Subplot 1: Log-Transformed Histogram showing frequency distribution
log_comp = np.log1p(df['ConvertedCompYearly'])  # Using log1p to handle zeros
axes[0].hist(log_comp, bins=30, color='skyblue', edgecolor='black')
axes[0].set_title('Log-Transformed Compensation Distribution', fontsize=14)
axes[0].set_xlabel('Log(Compensation + 1) (in $1000s)', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].grid(True, alpha=0.3)
axes[0].annotate(f'Skewness: {df["ConvertedCompYearly"].skew():.2f}', 
                xy=(0.05, 0.95), xycoords='axes fraction',
                fontsize=10, color='gray', ha='left')

# Subplot 2: Log-Transformed Distribution (Density) Plot showing the shape of distribution
sns.kdeplot(log_comp, ax=axes[1], fill=True, color='skyblue')
axes[1].set_title('Log-Transformed Distribution', fontsize=14)
axes[1].set_xlabel('Log(Compensation + 1) (in $1000s)', fontsize=12)
axes[1].set_ylabel('Density', fontsize=12)
axes[1].grid(True, alpha=0.3)

# Add statistical summary
def add_summary(ax, data_col, title):
    mean_val = data_col.mean()
    std_val = data_col.std()
    ax.annotate(f'Mean: ${mean_val:,.0f}\nStd: ${std_val:,.0f}', 
                xy=(0.05, 0.95), xycoords='axes fraction',
                fontsize=10, color='darkblue')

add_summary(axes[0], log_comp, 'Log-Transformed Distribution')
add_summary(axes[1], log_comp, 'Log-Transformed Distribution')

plt.tight_layout()
plt.show()

<h3>Step 4: Calculate Median Compensation for Full-Time Employees</h3>


**Task**: Filter the data to calculate the median compensation for respondents whose employment status is "Employed, full-time."


In [ ]:
# ... existing code from previous steps ...

# Step 4: Calculate Median Compensation for Full-Time Employees
full_time = df[df['Employment'] == 'Employed, full-time']
median_compensation = full_time['ConvertedCompYearly'].median()

print(f'Median compensation for full‑time employees: ${median_compensation:,.0f}')

<h3>Step 5: Analyzing Compensation Range and Distribution by Country</h3>


Explore the range of compensation in the ConvertedCompYearly column by analyzing differences across countries. Use box plots to compare the compensation distributions for each country to identify variations and anomalies within each region, providing insights into global compensation trends.



In [ ]:
"""
Simple inspection of available columns and data types
"""

# Clean data - remove extra spaces from Country column (if it exists)
df['Country'] = df['Country'].str.strip() if 'Country' in df.columns else df['Country'].str.strip()

print("=" * 70)
print("DATASET INFORMATION")
print("=" * 70)

# Basic information
print(f"\nDataset Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

# Column names with data types
print("\n--- COLUMN NAMES AND DATA TYPES ---")
for col in df.columns:
    print(f"• {col}: {df[col].dtype}")

# Quick stats for each column
print("\n--- QUICK STATISTICS ---")
for col in df.columns:
    count = df[col].count()
    nulls = df[col].isnull().sum()
    
    if pd.api.types.is_numeric_dtype(df[col]):
        print(f"\n{col}:")
        print(f"  Count: {count}, Nulls: {nulls}")
        print(f"  Min: {df[col].min():,.0f} | Max: {df[col].max():,.0f}")
    else:
        top_5 = df[df[col].notna()][col].value_counts().head(3)
        print(f"\n{col}:")
        print(f"  Count: {count}, Nulls: {nulls}")
        print(f"  Top values:\n{top_5.to_string()}")

print("\n" + "=" * 70)

In [ ]:

# --- Step 5: Analyzing Compensation by Country ---

# Cap outliers and calculate log
cap = 2_000_000
df_clean['CappedComp'] = df_clean['ConvertedCompYearly'].clip(upper=cap)
df_clean['LogComp'] = np.log1p(df_clean['CappedComp'])

# Identify top-10 countries
num_unique_countries = len(df_clean['Country'].unique())
top_10_sum = (
    df_clean.groupby('Country')['CappedComp']
    .sum()
    .nlargest(min(10, num_unique_countries))
    .reset_index(name='Total_Comp')
)

top_10_mean = (
    df_clean.groupby('Country')['CappedComp']
    .mean()
    .nlargest(min(10, num_unique_countries))
    .reset_index(name='Mean_Comp')
)

# --- Create Dual Plot ---
fig, axes = plt.subplots(2, 1, figsize=(14, 12))

# Define color palette
palette = sns.color_palette("Set2", len(top_10_sum))
country_color_map = {country: palette[i] for i, country in enumerate(top_10_sum['Country'])}

# --- Top Panel: Box Plot ---
# Added hue and legend=False to fix the FutureWarning
sns.boxplot(
    data=df_clean[df_clean['Country'].isin(top_10_sum['Country'])],
    x='LogComp',
    y='Country',
    hue='Country',
    ax=axes[0],
    order=top_10_sum['Country'],
    palette=palette,
    legend=False,
    linewidth=2.5
)

axes[0].set_title('Top 10 Countries by Total Compensation (Log Scale Distribution)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Log(Compensation + 1)')

# Add mean markers
for idx, country in enumerate(top_10_sum['Country']):
    mean_val = df_clean.loc[df_clean['Country'] == country, 'CappedComp'].mean()
    log_mean = np.log1p(mean_val)
    color = country_color_map[country]
    axes[0].axvline(log_mean, color=color, linestyle='--', alpha=0.6, linewidth=1.5)
    axes[0].plot(log_mean, idx, 'o', color=color, markersize=8, markerfacecolor='white', markeredgewidth=1.5)

# --- Bottom Panel: Bar Plot ---
# Added hue and legend=False to fix the FutureWarning
sns.barplot(
    data=top_10_mean,
    x='Mean_Comp',
    y='Country',
    hue='Country',
    ax=axes[1],
    palette=palette,
    legend=False
)

# Overlay KDE
ax2_twin = axes[1].twiny()
sns.kdeplot(df_clean['CappedComp'], color='red', ax=ax2_twin, fill=True, alpha=0.1)
ax2_twin.set_xlabel('Global Density (Red Shadow)', color='red')

axes[1].set_title('Top 10 Countries by Mean Compensation (Average $)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Annual Compensation ($)')

# ... (rest of your plotting code above remains the same)

# --- Legend Construction ---
country_legend = [
    plt.Line2D([], [], color=c, lw=3, label=country) 
    for country, c in country_color_map.items()
] + [
    plt.Line2D([], [], color='red', lw=2, label='Global Skewed Dist.')
]

# PLACE UNDER THE GRAPH:
# We attach it to axes[1] (the bottom plot)
# ncol=3 or 4 spreads the labels into columns so it doesn't take much vertical space
axes[1].legend(
    handles=country_legend, 
    loc='upper center', 
    bbox_to_anchor=(0.5, -0.2),  # (Horizontal center, Vertical space below)
    ncol=3,                      # 3 columns of names
    fontsize=10, 
    frameon=True
)

# ADJUST LAYOUT:
# We add a little extra padding at the bottom (bottom=0.15) 
# so the legend doesn't get cut off.
plt.tight_layout()
plt.subplots_adjust(bottom=0.15) 

plt.show()

<h3>Step 6: Removing Outliers from the Dataset</h3>


**Task**: Create a new DataFrame by removing outliers from the `ConvertedCompYearly` column to get a refined dataset for correlation analysis.


In [ ]:
## Write your code here
# ... existing code from previous steps ...

# Step 6: Removing Outliers from the Dataset
# Calculate Q1 (25th percentile) and Q3 (75th percentile)
Q1 = df['ConvertedCompYearly'].quantile(0.25)
Q3 = df['ConvertedCompYearly'].quantile(0.75)
IQR = Q3 - Q1

# Define the lower and upper bounds for outliers
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Create a new DataFrame without outliers
df_no_outliers = df[(df['ConvertedCompYearly'] >= lower_bound) & (df['ConvertedCompYearly'] <= upper_bound)]

# Display the shape to show how many rows were removed
print(f"Original dataset shape: {df.shape}")
print(f"Dataset shape after removing outliers: {df_no_outliers.shape}")

<h3>Step 7: Finding Correlations Between Key Variables</h3>


**Task**: Calculate correlations between `ConvertedCompYearly`, `WorkExp`, and `JobSatPoints_1`. Visualize these correlations with a heatmap.


In [ ]:
## Write your code here
# ... existing code from previous steps ...

# Step 7: Finding Correlations Between Key Variables
# Select the relevant columns
correlation_data = df_no_outliers[['ConvertedCompYearly', 'WorkExp', 'JobSatPoints_1']]

# Calculate the correlation matrix
correlation_matrix = correlation_data.corr()

# Visualize the correlations with a heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5)
plt.title('Correlation Matrix: Compensation, Work Experience, and Job Satisfaction', fontsize=14)
plt.show()

<h3>Step 8: Scatter Plot for Correlations</h3>


**Task**: Create scatter plots to examine specific correlations between `ConvertedCompYearly` and `WorkExp`, as well as between `ConvertedCompYearly` and `JobSatPoints_1`.


In [ ]:
## Write your code here
# ... existing code from previous steps ...

# Step 8: Scatter Plot for Correlations
# Create a figure with two subplots (side by side)
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Scatter plot 1: ConvertedCompYearly vs. WorkExp
axes[0].scatter(df_no_outliers['WorkExp'], df_no_outliers['ConvertedCompYearly'], alpha=0.5)
axes[0].set_title('Compensation vs. Work Experience', fontsize=14)
axes[0].set_xlabel('Years of Work Experience')
axes[0].set_ylabel('Annual Compensation ($)')
axes[0].grid(True, alpha=0.3)

# Scatter plot 2: ConvertedCompYearly vs. JobSatPoints_1
axes[1].scatter(df_no_outliers['JobSatPoints_1'], df_no_outliers['ConvertedCompYearly'], alpha=0.5)
axes[1].set_title('Compensation vs. Job Satisfaction', fontsize=14)
axes[1].set_xlabel('Job Satisfaction Points')
axes[1].set_ylabel('Annual Compensation ($)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

<h3>Summary</h3>


In this lab, you practiced essential skills in correlation analysis by:

- Examining the distribution of yearly compensation with histograms and box plots.
- Detecting and removing outliers from compensation data.
- Calculating correlations between key variables such as compensation, work experience, and job satisfaction.
- Visualizing relationships with scatter plots and heatmaps to gain insights into the associations between these features.

By following these steps, you have developed a solid foundation for analyzing relationships within the dataset.


## Authors:
Ayushi Jain


### Other Contributors:
- Rav Ahuja
- Lakshmi Holla
- Malika


Copyright © IBM Corporation. All rights reserved.
